<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/03_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Loading the data and making the checklists**

In [1]:
!pip install -q datasets scikit-learn

from datasets import load_dataset
import numpy as np

ds = load_dataset("coastalcph/lex_glue", "unfair_tos")
names = ds["train"].features["labels"].feature.names

def to_checklist(labels_list):
    """Turn label lists like [2, 4] into 8-box checklists like [0,0,1,0,1,0,0,0]."""
    y = np.zeros((len(labels_list), 8))
    for i, labs in enumerate(labels_list):
        for l in labs:
            y[i, l] = 1
    return y

X_train, y_train = ds["train"]["text"], to_checklist(ds["train"]["labels"])
X_val, y_val = ds["validation"]["text"], to_checklist(ds["validation"]["labels"])
X_test, y_test = ds["test"]["text"], to_checklist(ds["test"]["labels"])

print("Train:", len(X_train), "sentences, checklist shape:", y_train.shape)
print("Total ticked boxes in train:", int(y_train.sum()))

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

unfair_tos/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  501kB            

unfair_tos/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

unfair_tos/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147kB            

unfair_tos/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

unfair_tos/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1607 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2275 [00:00<?, ? examples/s]

Train: 5532 sentences, checklist shape: (5532, 8)
Total ticked boxes in train: 702


### **TF-IDF**
turns each sentence into word-importance scores, the numbers the SVM will learn from.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 3))     # single words, pairs and triples

X_train_vec = tfidf.fit_transform(X_train)      # learn the word list from training, then score
X_val_vec = tfidf.transform(X_val)              # score with the same list
X_test_vec = tfidf.transform(X_test)

print("Training matrix shape:", X_train_vec.shape)

# Look at the top-scoring words in one sentence
feature_names = tfidf.get_feature_names_out()
row = X_train_vec[10]
print("\nSentence:", X_train[10])
for i in row.toarray()[0].argsort()[::-1][:8]:
    print(f"  {feature_names[i]}: {row[0, i]:.3f}")

Training matrix shape: (5532, 132543)

Sentence: we may make changes to this agreement and to the services from time to time . 

  services from time: 0.242
  changes to this: 0.242
  agreement and to: 0.231
  may make changes: 0.231
  the services from: 0.223
  we may make: 0.217
  services from: 0.212
  time: 0.211


### **Training the 8 SVM's**

**C:** It controls how strictly the SVM tries to get every training sentence right:

**Too strict (high C):** it memorizes the training sentences, including their quirks and does worse on new ones

**Too relaxed (low C):** it misses real patterns

In [6]:
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score

best_c, best_score = None, -1

for c in [10]:
    model = OneVsRestClassifier(LinearSVC(C=c))     # 8 SVMs, one per type
    model.fit(X_train_vec, y_train)                 # learn from training data
    pred = model.predict(X_val_vec)                 # answer the practice exam
    score = f1_score(y_val, pred, average="macro", zero_division=0)
    print(f"C = {c}: validation macro F1 = {score:.3f}")
    if score > best_score:
        best_c, best_score, svm = c, score, model   # keep the best one

print("\nBest C:", best_c)

C = 10: validation macro F1 = 0.656

Best C: 10


I tested C values from 0.1 to 100 on the validation set. The score rose up to C = 10 and then levelled off (0.656 at C = 10 and 30). C = 100 scored slightly higher (0.662) but did not converge, so I chose C = 10.

### **The scoring function**

In [7]:
def lexglue_scores(y_true, y_pred):
    """Micro and macro F1 the LexGLUE way: add a 9th 'none' box for sentences with no unfair type."""
    def add_none(y):
        y = np.asarray(y)
        none = (y.sum(axis=1) == 0).astype(int).reshape(-1, 1)   # 1 if no type ticked
        return np.hstack([y, none])                               # 8 boxes + 'none' box
    t, p = add_none(y_true), add_none(y_pred)
    micro = f1_score(t, p, average="micro", zero_division=0) * 100
    macro = f1_score(t, p, average="macro", zero_division=0) * 100
    return micro, macro

# Test it on the validation set
micro, macro = lexglue_scores(y_val, svm.predict(X_val_vec))
print(f"Validation: micro F1 = {micro:.1f}, macro F1 = {macro:.1f}")

Validation: micro F1 = 94.4, macro F1 = 69.1


### **Final scores on the test set**
**Three models:**

**The SVM**

**“Always fair”:** predicts “nothing unfair” for every sentence

**“Random”:** guesses each type randomly, as often as that type appears in the training data

In [8]:
# 1. Our SVM
svm_micro, svm_macro = lexglue_scores(y_test, svm.predict(X_test_vec))

# 2. Dumb model: always says "fair"
fair_micro, fair_macro = lexglue_scores(y_test, np.zeros_like(y_test))

# 3. Dumb model: random guesses, as often as each type appears in training
np.random.seed(42)
rates = y_train.mean(axis=0)                                   # how common each type is
random_pred = (np.random.rand(*y_test.shape) < rates).astype(int)
rand_micro, rand_macro = lexglue_scores(y_test, random_pred)

print(f"{'Model':<22}{'Micro F1':>10}{'Macro F1':>10}")
print(f"{'Always fair':<22}{fair_micro:>10.1f}{fair_macro:>10.1f}")
print(f"{'Random':<22}{rand_micro:>10.1f}{rand_macro:>10.1f}")
print(f"{'TF-IDF + SVM (ours)':<22}{svm_micro:>10.1f}{svm_macro:>10.1f}")
print(f"{'TF-IDF + SVM (paper)':<22}{94.7:>10.1f}{75.0:>10.1f}")

Model                   Micro F1  Macro F1
Always fair                 88.9      10.5
Random                      78.2      10.8
TF-IDF + SVM (ours)         95.3      76.5
TF-IDF + SVM (paper)        94.7      75.0


On the test set, my TF-IDF + SVM baseline scored 95.3 micro F1 and 76.5 macro F1, closely matching the LexGLUE paper result (94.7 / 75.0). An ‘always fair’ model reached 88.9 micro F1 but only 10.5 macro F1, showing why micro F1 alone is misleading on this imbalanced data.

### **Score per type**

shows the F1 score for each unfair type separately on the test set, plus how many examples of each type the test set has. This tells which types the SVM handles well and which it struggles with.

In [9]:
pred = svm.predict(X_test_vec)
per_type = f1_score(y_test, pred, average=None, zero_division=0)   # one score per type
test_counts = y_test.sum(axis=0)                                    # examples per type in test

print(f"{'Type':<26}{'F1':>6}   Test examples")
for name, score, n in zip(names, per_type, test_counts):
    print(f"{name:<26}{score * 100:>6.1f}   {int(n)}")

Type                          F1   Test examples
Limitation of liability     77.6   38
Unilateral termination      66.7   38
Unilateral change           63.5   38
Content removal             76.9   13
Contract by using           80.0   23
Choice of law               91.7   13
Jurisdiction                74.1   16
Arbitration                 60.0   7


per-type F1 scores shows that rare types arent always the hardest. Choice of law scored highest (91.7) because its wording is formulaic, while Unilateral change (63.5) and Unilateral termination (66.7) scored lower because they are phrased in many different ways. Arbitration (60.0) has only 7 test examples, so its score is unstable.